# 00 — Statistical Assumptions & Mathematical Foundations

This notebook examines the mathematical assumptions underlying both the heuristic Markov chain and the Gaussian HMM applied to our LP regime detection problem.

**Problem:** We model hourly regime dynamics of a Uniswap V3 ETH/USDC 0.05% liquidity pool using a Hidden Markov Model. The observations are 4-dimensional hourly feature vectors (log-return, swap count, total fees, realised volatility). The latent states represent LP-relevant market regimes (Goldilocks, Trending, Toxic).

**What we test here:**

| # | Assumption | Why it matters |
|---|------------|----------------|
| 1 | **Stationarity** of feature series | Time-homogeneous transition matrix $A_{ij}$ requires the data-generating process to not drift |
| 2 | **Gaussian emission** $X_t \mid Z_t=k \sim \mathcal{N}(\mu_k, \Sigma_k)$ | EM algorithm's M-step and BIC validity depend on this |
| 3 | **Conditional independence** of features given state | Diagonal covariance `diag` assumption in `hmmlearn` |
| 4 | **Order-1 Markov property** $P(Z_t \mid Z_{t-1}, \ldots) = P(Z_t \mid Z_{t-1})$ | The entire chain structure rests on this |
| 5 | **Time-homogeneity** of $A$ | Transition probabilities assumed constant across the full sample |
| 6 | **Ergodicity** and stationary distribution $\pi$ | Required for long-run interpretability and mean sojourn times |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from pathlib import Path
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

LABELLED_PATH = Path('../data/hourly_features_labelled.parquet')
FEATURE_COLS  = ['log_return', 'swap_count', 'total_fees_usd', 'realised_vol']
STATE_NAMES   = ['Goldilocks', 'Trending', 'Toxic']
STATE_COLORS  = {'Goldilocks': '#2ecc71', 'Trending': '#f39c12', 'Toxic': '#e74c3c'}

df = pd.read_parquet(LABELLED_PATH)
print(f'Loaded {len(df):,} hourly bars with columns: {list(df.columns)}')

---
## 1. Stationarity of Feature Series

A time-homogeneous Markov chain assumes that the transition matrix $A$ is constant:

$$P(Z_t = j \mid Z_{t-1} = i) = A_{ij}, \quad \forall t$$

If the *observable* features exhibit non-stationarity (unit roots, trends, structural breaks), the latent process driving them is unlikely to be time-homogeneous either.

We apply two complementary tests per feature:

- **Augmented Dickey-Fuller (ADF):** $H_0$: unit root (non-stationary). Rejection → evidence of stationarity.
- **KPSS:** $H_0$: trend-stationary. Rejection → evidence of non-stationarity.

The two tests have *opposite* nulls, so their joint interpretation is:

| ADF rejects | KPSS rejects | Conclusion |
|:-----------:|:------------:|:----------:|
| Yes | No | **Stationary** ✓ |
| No | Yes | **Non-stationary** ✗ |
| Yes | Yes | Trend-stationary (ambiguous) |
| No | No | Low power / inconclusive |

In [ ]:
def stationarity_tests(series, name, alpha=0.05):
    """Run ADF and KPSS on a univariate series and return a summary dict."""
    
    # ADF test: H0 = unit root (non-stationary)
    adf_stat, adf_p, adf_lags, _, _, _ = adfuller(series.dropna(), autolag='AIC')
    
    # KPSS test: H0 = trend-stationary
    # regression='c' tests level-stationarity (no deterministic trend)
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(series.dropna(), regression='c', nlags='auto')
    
    # Joint interpretation
    adf_reject  = adf_p < alpha
    kpss_reject = kpss_p < alpha
    
    if adf_reject and not kpss_reject:
        verdict = 'Stationary ✓'
    elif not adf_reject and kpss_reject:
        verdict = 'Non-stationary ✗'
    elif adf_reject and kpss_reject:
        verdict = 'Trend-stationary (ambiguous)'
    else:
        verdict = 'Inconclusive'
    
    return {
        'Feature': name,
        'ADF stat': round(adf_stat, 4),
        'ADF p-value': round(adf_p, 4),
        'ADF rejects H0': adf_reject,
        'KPSS stat': round(kpss_stat, 4),
        'KPSS p-value': round(kpss_p, 4),
        'KPSS rejects H0': kpss_reject,
        'Verdict': verdict
    }

# Run both tests on each feature
stationarity_results = [stationarity_tests(df[col], col) for col in FEATURE_COLS]
stationarity_df = pd.DataFrame(stationarity_results).set_index('Feature')

print('Stationarity tests (α = 0.05):')
display(stationarity_df)

In [ ]:
# Visual: rolling mean and std of each feature (window = 500 hours ≈ 3 weeks)
# If the series is stationary, the rolling statistics should be roughly flat.

ROLLING_WINDOW = 500

fig, axes = plt.subplots(len(FEATURE_COLS), 1, figsize=(16, 3 * len(FEATURE_COLS)), sharex=True)

for ax, col in zip(axes, FEATURE_COLS):
    series = df[col]
    rolling_mean = series.rolling(ROLLING_WINDOW).mean()
    rolling_std  = series.rolling(ROLLING_WINDOW).std()
    
    ax.plot(df['window'], series, lw=0.2, alpha=0.4, color='grey', label='raw')
    ax.plot(df['window'], rolling_mean, lw=1.5, color='blue', label=f'rolling mean ({ROLLING_WINDOW}h)')
    ax.plot(df['window'], rolling_std, lw=1.5, color='red', ls='--', label=f'rolling std ({ROLLING_WINDOW}h)')
    ax.set_ylabel(col, fontsize=9)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Window (hourly index)')
fig.suptitle('Rolling statistics — visual stationarity check', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation notes:**

- `log_return` is expected to be stationary (returns are typically I(0)).
- `swap_count` and `total_fees_usd` may show non-stationarity if pool activity grew over time (adoption trend).
- `realised_vol` may show volatility clustering (ARCH effects) — stationary in level but with time-varying variance.
- If a feature is non-stationary, the HMM's time-homogeneity assumption is violated for that feature's emission distribution. This doesn't invalidate the model entirely, but it means regime parameters ($\mu_k, \Sigma_k$) estimated on the full sample may not represent any single period well.

---
## 2. Gaussian Emission Assumption

The Gaussian HMM assumes each state $k$ emits observations from a multivariate normal:

$$X_t \mid Z_t = k \sim \mathcal{N}(\mu_k, \Sigma_k)$$

The EM algorithm's M-step computes:

$$\hat{\mu}_k = \frac{\sum_t \gamma_t(k)\, x_t}{\sum_t \gamma_t(k)}, \quad \hat{\Sigma}_k = \frac{\sum_t \gamma_t(k)\,(x_t - \hat{\mu}_k)(x_t - \hat{\mu}_k)^\top}{\sum_t \gamma_t(k)}$$

where $\gamma_t(k) = P(Z_t = k \mid X_{1:T})$ are the posterior state probabilities.

These are MLEs **under the Gaussian assumption**. If the true emission is heavy-tailed (e.g. Student-$t$), $\hat{\mu}_k$ is still consistent (Gaussian MLE coincides with method of moments for the first two moments), but:
- The likelihood score is misspecified → **BIC is no longer a valid model selector**
- The posterior $\gamma_t(k)$ will underweight outlier observations, leading to state misclassification at extreme events

We test normality per feature, both globally and per heuristic state, using:
- **Jarque-Bera:** tests skewness = 0 and excess kurtosis = 0 jointly. $H_0$: Gaussian.
- **Excess kurtosis:** $\kappa = \frac{E[(X-\mu)^4]}{\sigma^4} - 3$. If $\kappa \gg 0$, tails are heavier than Gaussian.

In [ ]:
# --- 2a. Global normality tests ---

def normality_summary(series, name):
    """Compute Jarque-Bera test, skewness, and excess kurtosis."""
    s = series.dropna()
    jb_stat, jb_p = stats.jarque_bera(s)
    return {
        'Feature': name,
        'N': len(s),
        'Skewness': round(float(stats.skew(s)), 4),
        'Excess kurtosis': round(float(stats.kurtosis(s)), 4),  # scipy uses Fisher def (excess)
        'JB stat': round(jb_stat, 2),
        'JB p-value': f'{jb_p:.2e}',
        'Gaussian?': 'Yes' if jb_p > 0.05 else 'No ✗'
    }

norm_results = [normality_summary(df[col], col) for col in FEATURE_COLS]
norm_df = pd.DataFrame(norm_results).set_index('Feature')

print('Global normality tests (α = 0.05):')
display(norm_df)

In [ ]:
# --- 2b. Per-state normality tests ---
# The HMM emission is conditioned on state, so the relevant test is
# whether features are Gaussian WITHIN each state, not globally.

per_state_results = []
for state in STATE_NAMES:
    mask = df['heuristic_state'] == state
    for col in FEATURE_COLS:
        res = normality_summary(df.loc[mask, col], col)
        res['State'] = state
        per_state_results.append(res)

per_state_df = pd.DataFrame(per_state_results)[['State', 'Feature', 'Skewness', 'Excess kurtosis', 'JB p-value', 'Gaussian?']]

print('Per-state normality tests:')
display(per_state_df.set_index(['State', 'Feature']))

In [ ]:
# --- 2c. QQ-plots per feature (global) ---
# If the data is Gaussian, points lie on the diagonal.
# Heavy tails show as S-shaped deviations at the extremes.

fig, axes = plt.subplots(1, len(FEATURE_COLS), figsize=(16, 4))

for ax, col in zip(axes, FEATURE_COLS):
    data = df[col].dropna().values
    # Standardise for comparable QQ plotting
    z = (data - data.mean()) / data.std()
    
    theoretical = np.sort(stats.norm.ppf(np.linspace(0.001, 0.999, len(z))))
    empirical   = np.sort(z)
    
    # Subsample for plotting if too many points
    step = max(1, len(z) // 2000)
    ax.scatter(theoretical[::step], empirical[::step], s=3, alpha=0.5, color='steelblue')
    
    lim = max(abs(theoretical.min()), abs(theoretical.max()), abs(empirical.min()), abs(empirical.max()))
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='N(0,1) reference')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Empirical quantiles')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle('QQ-plots against standard normal', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation guidance:**

- `log_return`: expected to show heavy tails ($\kappa \gg 3$) — this is a well-known stylised fact of financial returns. A Student-$t$ emission would be more appropriate but `hmmlearn` does not support it natively.
- `swap_count`: a count variable is discrete and bounded below by 0 — Gaussian is a rough approximation. A Poisson or negative binomial emission would be more natural.
- `total_fees_usd` and `realised_vol`: both non-negative and right-skewed. A log-transform before standardisation would bring them closer to Gaussian.

The Gaussian assumption is almost certainly rejected for all features. This is expected and does not invalidate the HMM exercise — but it means:
1. BIC comparisons should be treated as heuristic, not exact
2. State boundaries near the tails may be unreliable
3. A more sophisticated emission model (mixture of Gaussians per state, or Student-$t$) could improve fit

---
## 3. Feature Conditional Independence (Diagonal Covariance)

Our HMM uses `covariance_type='diag'`, which assumes features are **conditionally independent** given the hidden state:

$$\Sigma_k = \text{diag}(\sigma^2_{k,1}, \ldots, \sigma^2_{k,d})$$

This means:
$$P(X_t \mid Z_t = k) = \prod_{j=1}^{d} \mathcal{N}(x_{t,j};\, \mu_{k,j},\, \sigma^2_{k,j})$$

If features are correlated within a state, the diagonal model:
- Loses information about covariance structure → worse state discrimination
- The log-likelihood is lower than it should be → BIC favours more states to compensate

We test this with:
- **Pearson correlation matrices** per state
- **Bartlett's test of sphericity**: $H_0$: correlation matrix = identity (features independent). Rejection → features are correlated.

Parameter count comparison:
- Diagonal: $k_{\text{diag}} = n \times d = 3 \times 4 = 12$ covariance params
- Full: $k_{\text{full}} = n \times \frac{d(d+1)}{2} = 3 \times 10 = 30$ covariance params

In [ ]:
# --- 3a. Per-state correlation matrices ---

fig, axes = plt.subplots(1, len(STATE_NAMES), figsize=(5 * len(STATE_NAMES), 4))

for ax, state in zip(axes, STATE_NAMES):
    mask = df['heuristic_state'] == state
    corr = df.loc[mask, FEATURE_COLS].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, linewidths=0.5,
                xticklabels=[c[:10] for c in FEATURE_COLS],
                yticklabels=[c[:10] for c in FEATURE_COLS])
    ax.set_title(f'{state} (n={mask.sum():,})', fontsize=10)
    ax.tick_params(axis='both', labelsize=7)

fig.suptitle('Per-state Pearson correlation matrices', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3b. Bartlett's test of sphericity per state ---
# H0: the correlation matrix is the identity matrix (features are uncorrelated)
# Test statistic: -[(n-1) - (2d+5)/6] * ln|R|  ~ chi2(d*(d-1)/2)

def bartlett_sphericity(data):
    """Bartlett's test of sphericity.
    Returns (chi2_stat, p_value, degrees_of_freedom).
    """
    n, d = data.shape
    R = np.corrcoef(data, rowvar=False)
    
    # Log of determinant of correlation matrix
    sign, log_det = np.linalg.slogdet(R)
    
    # Bartlett statistic
    chi2_stat = -((n - 1) - (2 * d + 5) / 6) * log_det
    dof = d * (d - 1) // 2
    p_value = 1 - stats.chi2.cdf(chi2_stat, dof)
    
    return chi2_stat, p_value, dof

print('Bartlett\'s test of sphericity (H0: correlation matrix = I):\n')

bartlett_results = []
for state in STATE_NAMES:
    mask = df['heuristic_state'] == state
    X_state = df.loc[mask, FEATURE_COLS].dropna().values
    chi2_stat, p_val, dof = bartlett_sphericity(X_state)
    reject = p_val < 0.05
    bartlett_results.append({
        'State': state,
        'n': len(X_state),
        'χ² stat': round(chi2_stat, 2),
        'dof': dof,
        'p-value': f'{p_val:.2e}',
        'Reject H0 (α=0.05)': reject,
        'Implication': 'Features correlated → diag cov loses info' if reject else 'Features ~independent → diag cov OK'
    })

display(pd.DataFrame(bartlett_results).set_index('State'))

**Interpretation:**

If Bartlett rejects in all states, the diagonal covariance assumption discards meaningful cross-feature structure. However, with $T=42{,}301$ and $d=4$:
- A full-covariance model adds $30 - 12 = 18$ parameters (for 3 states)
- This is negligible relative to the sample size
- **Recommendation:** if Bartlett rejects, re-fit the HMM with `covariance_type='full'` and compare BIC

---
## 4. Testing the Markov Property (Order Selection)

The first-order Markov property states:

$$P(Z_t = j \mid Z_{t-1} = i,\, Z_{t-2} = l,\, \ldots) = P(Z_t = j \mid Z_{t-1} = i)$$

We test this using the **Billingsley (1961) likelihood ratio test** for order 1 vs order 2.

### Test construction

Let $n_{ijk}$ = number of transitions $i \to j \to k$ in the observed heuristic state sequence.

Under $H_0$ (first-order), the probability of $k$ given predecessor pair $(i,j)$ depends only on $j$:

$$P(Z_t = k \mid Z_{t-1} = j,\, Z_{t-2} = i) = P(Z_t = k \mid Z_{t-1} = j) = \frac{n_{jk}}{n_{j\cdot}}$$

Under $H_1$ (second-order), it depends on both:

$$P(Z_t = k \mid Z_{t-1} = j,\, Z_{t-2} = i) = \frac{n_{ijk}}{n_{ij\cdot}}$$

The likelihood ratio statistic is:

$$\Lambda = 2 \sum_{i,j,k} n_{ijk} \log \frac{n_{ijk} \,/\, n_{ij\cdot}}{n_{jk} \,/\, n_{j\cdot}}$$

Under $H_0$, $\Lambda \sim \chi^2(d(d-1)^2)$ where $d$ = number of states.

In [ ]:
def billingsley_order_test(state_sequence, state_names):
    """
    Billingsley (1961) likelihood ratio test: first-order vs second-order Markov.
    
    Parameters
    ----------
    state_sequence : array-like of state labels
    state_names    : list of unique state labels (defines ordering)
    
    Returns
    -------
    dict with test statistic, p-value, degrees of freedom
    """
    d = len(state_names)
    idx = {s: i for i, s in enumerate(state_names)}
    seq = [idx[s] for s in state_sequence]
    
    # Count 3-grams n_ijk
    n_ijk = np.zeros((d, d, d), dtype=float)
    for t in range(2, len(seq)):
        i, j, k = seq[t-2], seq[t-1], seq[t]
        n_ijk[i, j, k] += 1
    
    # Marginals
    n_ij_dot = n_ijk.sum(axis=2)  # n_{ij.} = sum over k
    n_jk     = n_ijk.sum(axis=0)  # n_{jk}  = sum over i
    n_j_dot  = n_jk.sum(axis=1)   # n_{j.}  = sum over k of n_{jk}
    
    # Compute likelihood ratio statistic
    Lambda = 0.0
    for i in range(d):
        for j in range(d):
            for k in range(d):
                if n_ijk[i,j,k] > 0 and n_ij_dot[i,j] > 0 and n_jk[j,k] > 0 and n_j_dot[j] > 0:
                    p_second_order = n_ijk[i,j,k] / n_ij_dot[i,j]   # P(k | i,j)
                    p_first_order  = n_jk[j,k] / n_j_dot[j]         # P(k | j)
                    Lambda += 2 * n_ijk[i,j,k] * np.log(p_second_order / p_first_order)
    
    # Degrees of freedom: d * (d-1)^2
    dof = d * (d - 1) ** 2
    p_value = 1 - stats.chi2.cdf(Lambda, dof)
    
    return {
        'Λ (LR statistic)': round(Lambda, 2),
        'dof': dof,
        'p-value': p_value,
        'Reject H0 (α=0.05)': p_value < 0.05,
        'n_ijk tensor shape': n_ijk.shape,
        'Total 3-grams': int(n_ijk.sum())
    }

state_seq = df['heuristic_state'].values
billingsley = billingsley_order_test(state_seq, STATE_NAMES)

print('Billingsley (1961) order test: H0 = first-order Markov vs H1 = second-order\n')
for k, v in billingsley.items():
    if k == 'p-value':
        print(f'  {k}: {v:.4e}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# --- 4b. Chapman-Kolmogorov consistency check ---
# If Z is truly first-order Markov with transition matrix A, then:
#   A^(2)_empirical  ≈  A^2  (matrix square)
# Discrepancy indicates memory beyond lag 1.

def estimate_transition_matrix(state_sequence, state_names):
    """Estimate row-stochastic transition matrix from a sequence of states."""
    d = len(state_names)
    idx = {s: i for i, s in enumerate(state_names)}
    counts = np.zeros((d, d))
    for t in range(1, len(state_sequence)):
        i = idx[state_sequence[t-1]]
        j = idx[state_sequence[t]]
        counts[i, j] += 1
    # Normalise rows
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid div-by-zero for empty rows
    return counts / row_sums

def estimate_k_step_transition(state_sequence, state_names, k=2):
    """Estimate the k-step transition matrix directly from data."""
    d = len(state_names)
    idx = {s: i for i, s in enumerate(state_names)}
    counts = np.zeros((d, d))
    for t in range(k, len(state_sequence)):
        i = idx[state_sequence[t-k]]
        j = idx[state_sequence[t]]
        counts[i, j] += 1
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return counts / row_sums

A1 = estimate_transition_matrix(state_seq, STATE_NAMES)
A2_theory   = A1 @ A1                                        # A^2 (predicted by order-1)
A2_empirical = estimate_k_step_transition(state_seq, STATE_NAMES, k=2)  # actual 2-step

discrepancy = A2_empirical - A2_theory
frobenius   = np.linalg.norm(discrepancy, 'fro')

print(f'Chapman-Kolmogorov discrepancy: ||A²_empirical - A²_theory||_F = {frobenius:.6f}\n')

# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['A² (from theory = A × A)', 'A² (empirical 2-step)', 'Discrepancy']
matrices = [A2_theory, A2_empirical, discrepancy]
cmaps = ['Blues', 'Blues', 'RdBu_r']

for ax, mat, title, cmap in zip(axes, matrices, titles, cmaps):
    vabs = max(abs(mat.min()), abs(mat.max()))
    kwargs = {'vmin': 0, 'vmax': 1} if cmap == 'Blues' else {'vmin': -vabs, 'vmax': vabs}
    sns.heatmap(pd.DataFrame(mat, index=STATE_NAMES, columns=STATE_NAMES),
                annot=True, fmt='.4f', cmap=cmap, ax=ax, linewidths=0.5, **kwargs)
    ax.set_title(title, fontsize=10)

fig.suptitle('Chapman-Kolmogorov consistency: $A^{(2)}_{emp}$ vs $\hat{A}^2$', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4c. Autocorrelation of the state sequence ---
# If the process is first-order Markov, autocorrelation should decay
# geometrically: ρ(h) ~ λ₂^h where λ₂ is the second-largest eigenvalue of A.
# Slow decay indicates memory beyond lag 1.

from statsmodels.tsa.stattools import acf

# Encode states as integers for autocorrelation
state_int = df['heuristic_state'].map({s: i for i, s in enumerate(STATE_NAMES)}).values
max_lag = 100  # ~100 hours = ~4 days

acf_values = acf(state_int, nlags=max_lag, fft=True)

# Theoretical geometric decay from the second eigenvalue of A
eigenvalues = np.sort(np.abs(np.linalg.eigvals(A1)))[::-1]
lambda2 = eigenvalues[1]  # second-largest eigenvalue
theoretical_acf = lambda2 ** np.arange(max_lag + 1)
# Scale theoretical to match empirical at lag 1
scale = acf_values[1] / theoretical_acf[1] if theoretical_acf[1] != 0 else 1
theoretical_acf_scaled = theoretical_acf * scale

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(max_lag + 1), acf_values, width=0.8, color='steelblue', alpha=0.7, label='Empirical ACF')
ax.plot(range(max_lag + 1), theoretical_acf_scaled, 'r-', lw=2,
        label=f'Theoretical decay: $\lambda_2^h$, $\lambda_2$={lambda2:.4f}')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Lag (hours)')
ax.set_ylabel('Autocorrelation')
ax.set_title('State sequence autocorrelation vs first-order Markov prediction')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Second eigenvalue of A: λ₂ = {lambda2:.4f}')
print(f'Implied half-life of autocorrelation: {-np.log(2)/np.log(lambda2):.1f} hours')

**Interpretation:**

- If the empirical ACF closely follows the $\lambda_2^h$ curve, the first-order Markov property holds well.
- If the empirical ACF decays *slower* than the theoretical curve, the state sequence has longer memory than order-1 can capture — typical for financial regimes driven by structural macro conditions.
- The Billingsley test provides a formal $p$-value; the ACF plot provides visual intuition for the same phenomenon.

---
## 5. Time-Homogeneity of the Transition Matrix

Even if the Markov property holds, the transition matrix may **change over time**. This is especially likely in crypto markets where structural shifts (e.g. DeFi summer, FTX collapse, ETF approval) alter the dynamics.

We test this in two ways:

### 5a. Rolling transition matrix
Estimate $\hat{A}$ on a sliding window and track diagonal elements (self-transition probabilities) over time.

### 5b. Structural break test (Chow-type)
Split the sample at the midpoint, estimate $\hat{A}^{(1)}$ and $\hat{A}^{(2)}$ on each half, and test whether they are statistically equal.

$$\Lambda = 2 \left[ \ell(\hat{A}^{(1)}) + \ell(\hat{A}^{(2)}) - \ell(\hat{A}_{\text{full}}) \right]$$

where $\ell(A) = \sum_{i,j} n_{ij} \log A_{ij}$ and $\Lambda \sim \chi^2(d(d-1))$.

In [ ]:
# --- 5a. Rolling transition matrix ---

ROLL_SIZE = 2000  # ~2000 hours ≈ 83 days
ROLL_STEP = 200   # slide by 200 hours

roll_starts = range(0, len(state_seq) - ROLL_SIZE, ROLL_STEP)
roll_diags  = {s: [] for s in STATE_NAMES}
roll_centers = []

for start in roll_starts:
    chunk = state_seq[start:start + ROLL_SIZE]
    A_local = estimate_transition_matrix(chunk, STATE_NAMES)
    for i, s in enumerate(STATE_NAMES):
        roll_diags[s].append(A_local[i, i])
    roll_centers.append(start + ROLL_SIZE // 2)

fig, ax = plt.subplots(figsize=(14, 4))
for s in STATE_NAMES:
    ax.plot(roll_centers, roll_diags[s], lw=1.5, color=STATE_COLORS[s], label=f'P({s}→{s})')

ax.set_xlabel('Window center (hourly index)')
ax.set_ylabel('Self-transition probability')
ax.set_title(f'Rolling self-transition probabilities (window = {ROLL_SIZE}h, step = {ROLL_STEP}h)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5b. Chow-type structural break test ---
# Split at midpoint and test equality of transition matrices.

def mc_log_likelihood(sequence, state_names, A):
    """Compute Markov chain log-likelihood: sum n_ij * log(A_ij)."""
    d = len(state_names)
    idx = {s: i for i, s in enumerate(state_names)}
    ll = 0.0
    for t in range(1, len(sequence)):
        i, j = idx[sequence[t-1]], idx[sequence[t]]
        if A[i, j] > 0:
            ll += np.log(A[i, j])
    return ll

mid = len(state_seq) // 2
seq1, seq2 = state_seq[:mid], state_seq[mid:]

A_full = estimate_transition_matrix(state_seq, STATE_NAMES)
A1_half = estimate_transition_matrix(seq1, STATE_NAMES)
A2_half = estimate_transition_matrix(seq2, STATE_NAMES)

ll_full = mc_log_likelihood(state_seq, STATE_NAMES, A_full)
ll_1    = mc_log_likelihood(seq1, STATE_NAMES, A1_half)
ll_2    = mc_log_likelihood(seq2, STATE_NAMES, A2_half)

d = len(STATE_NAMES)
Lambda_chow = 2 * (ll_1 + ll_2 - ll_full)
dof_chow = d * (d - 1)  # each half has d(d-1) free params; full has same → diff = d(d-1)
p_chow = 1 - stats.chi2.cdf(Lambda_chow, dof_chow)

print('Structural break test (Chow-type LR test at sample midpoint):')
print(f'  Λ = {Lambda_chow:.2f}')
print(f'  dof = {dof_chow}')
print(f'  p-value = {p_chow:.4e}')
print(f'  Reject H0 (α=0.05): {p_chow < 0.05}')
print()

if p_chow < 0.05:
    print('  → The transition matrix is NOT constant across the sample.')
    print('    This is expected: crypto regimes evolve with market structure over years.')
    print('    The HMM\'s time-homogeneity is violated; parameter estimates are sample averages')
    print('    that may not represent any single sub-period accurately.')
else:
    print('  → No evidence of structural break at the midpoint.')

# Show the two half-sample transition matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(axes, [A1_half, A2_half], ['First half', 'Second half']):
    sns.heatmap(pd.DataFrame(mat, index=STATE_NAMES, columns=STATE_NAMES),
                annot=True, fmt='.3f', cmap='Blues', ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(title, fontsize=10)
fig.suptitle('Transition matrices: first half vs second half', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Ergodicity, Stationary Distribution, and Sojourn Times

A finite-state Markov chain has a unique stationary distribution $\pi$ if it is **irreducible** (all states communicate) and **aperiodic**. Then:

$$\pi = \pi A, \quad \sum_i \pi_i = 1$$

$\pi$ can be found as the **left eigenvector** of $A$ corresponding to eigenvalue 1.

The **mean sojourn time** (expected duration of a stay) in state $k$ is:

$$\mathbb{E}[\tau_k] = \frac{1}{1 - A_{kk}}$$

This has a direct LP interpretation: if $\mathbb{E}[\tau_{\text{Toxic}}] = 4$ hours, a Toxic regime is a short-lived spike; if it's 20 hours, it's a sustained danger period.

In [ ]:
# --- 6a. Irreducibility check ---
# A chain is irreducible iff the transition matrix (or a power of it) has all positive entries.
# With a finite number of states, we check if A^d has all entries > 0.

A = estimate_transition_matrix(state_seq, STATE_NAMES)
d = len(STATE_NAMES)

A_power = np.linalg.matrix_power(A, d)
irreducible = np.all(A_power > 0)
print(f'A^{d} has all positive entries: {irreducible}')
print(f'→ Chain is {"irreducible ✓" if irreducible else "NOT irreducible ✗"}')

# Aperiodicity: sufficient condition — at least one diagonal of A > 0
aperiodic = np.any(np.diag(A) > 0)
print(f'At least one self-loop (A_kk > 0): {aperiodic}')
print(f'→ Chain is {"aperiodic ✓" if aperiodic else "possibly periodic"}')

In [ ]:
# --- 6b. Stationary distribution via eigenvector ---
# π = πA  →  π is the left eigenvector of A with eigenvalue 1
# Equivalently, π^T is the right eigenvector of A^T with eigenvalue 1

eigenvalues_A, eigenvectors_A = np.linalg.eig(A.T)

# Find the eigenvector corresponding to eigenvalue ≈ 1
idx_one = np.argmin(np.abs(eigenvalues_A - 1.0))
pi_raw = np.real(eigenvectors_A[:, idx_one])
pi = pi_raw / pi_raw.sum()  # normalise to sum to 1

# Empirical occupancy for comparison
empirical_occupancy = df['heuristic_state'].value_counts(normalize=True).reindex(STATE_NAMES).values

stat_dist_df = pd.DataFrame({
    'State': STATE_NAMES,
    'π (theoretical)': np.round(pi, 4),
    'Empirical occupancy': np.round(empirical_occupancy, 4),
    'Discrepancy': np.round(pi - empirical_occupancy, 4)
}).set_index('State')

print('Stationary distribution vs empirical occupancy:')
display(stat_dist_df)

print(f'\nVerification: π sums to {pi.sum():.6f}')
print(f'Verification: πA = π? Max absolute error: {np.max(np.abs(pi @ A - pi)):.2e}')

In [ ]:
# --- 6c. Mean sojourn times: theoretical vs empirical ---
# Theoretical: E[τ_k] = 1 / (1 - A_kk)
# Empirical:   average run length of consecutive state-k labels

def compute_run_lengths(state_sequence, target_state):
    """Compute the length of each consecutive run of target_state."""
    runs = []
    count = 0
    for s in state_sequence:
        if s == target_state:
            count += 1
        else:
            if count > 0:
                runs.append(count)
            count = 0
    if count > 0:
        runs.append(count)
    return np.array(runs)

sojourn_results = []
for i, s in enumerate(STATE_NAMES):
    theoretical = 1 / (1 - A[i, i]) if A[i, i] < 1 else np.inf
    runs = compute_run_lengths(state_seq, s)
    empirical_mean = runs.mean() if len(runs) > 0 else 0
    empirical_median = np.median(runs) if len(runs) > 0 else 0
    
    sojourn_results.append({
        'State': s,
        'A_kk': round(A[i, i], 4),
        'E[τ] theoretical (hours)': round(theoretical, 2),
        'E[τ] empirical (hours)': round(empirical_mean, 2),
        'Median τ empirical (hours)': round(empirical_median, 2),
        'Num. runs': len(runs)
    })

sojourn_df = pd.DataFrame(sojourn_results).set_index('State')
print('Sojourn time analysis:')
display(sojourn_df)

In [ ]:
# --- 6d. Run-length distributions with theoretical geometric overlay ---
# Under the Markov model, sojourn time in state k is Geometric(1 - A_kk):
#   P(τ_k = t) = A_kk^{t-1} * (1 - A_kk)

fig, axes = plt.subplots(1, len(STATE_NAMES), figsize=(5 * len(STATE_NAMES), 4))

for ax, (i, s) in zip(axes, enumerate(STATE_NAMES)):
    runs = compute_run_lengths(state_seq, s)
    if len(runs) == 0:
        ax.set_title(f'{s}: no runs found')
        continue
    
    # Empirical histogram
    max_len = int(np.percentile(runs, 99))  # clip at 99th pct for readability
    bins = np.arange(1, max_len + 2) - 0.5
    ax.hist(runs, bins=bins, density=True, color=STATE_COLORS[s], alpha=0.6,
            edgecolor='white', label='Empirical')
    
    # Theoretical geometric PMF
    t_vals = np.arange(1, max_len + 1)
    p_exit = 1 - A[i, i]
    geom_pmf = (A[i, i] ** (t_vals - 1)) * p_exit
    ax.plot(t_vals, geom_pmf, 'k-o', markersize=3, lw=1.5,
            label=f'Geom(p={p_exit:.3f})')
    
    ax.set_title(f'{s}', fontsize=10)
    ax.set_xlabel('Run length (hours)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Sojourn time distributions: empirical vs Geometric (Markov prediction)', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation guidance:**

- If the empirical run-length histogram matches the geometric overlay, the Markov sojourn model is adequate.
- If empirical runs are **heavier-tailed** than the geometric (more very long runs), the regimes are more persistent than the Markov model predicts — indicative of long memory.
- For LP applications: if Toxic regimes have a heavy-tailed sojourn distribution, a simple "wait for regime change" strategy is riskier than the Markov model suggests.

---
## Summary of Findings

| Assumption | Test | Expected result | Implication if violated |
|:-----------|:-----|:----------------|:------------------------|
| Stationarity | ADF + KPSS | `log_return` stationary; activity features may drift | Time-varying emission params; consider sub-sampling or detrending |
| Gaussian emission | JB + QQ | Rejected for all features (heavy tails, skewness) | BIC approximate; extreme-event misclassification; consider Student-$t$ |
| Conditional independence | Bartlett | Likely rejected (fees ↔ swap count correlated) | Switch to `covariance_type='full'` |
| Order-1 Markov | Billingsley LR + ACF | Likely rejected at α=0.05 with 42k observations | Higher-order effects exist; practical impact moderated by persistence |
| Time-homogeneity | Rolling $A$ + Chow test | Likely rejected (multi-year crypto data spans structural shifts) | Regime dynamics are non-stationary; window-based estimation more robust |
| Ergodicity | Eigenvector $\pi$ vs occupancy | $\pi \approx$ empirical if long enough sample | If very different, chain hasn't mixed — or stationarity is violated |